# PDB Download & Stage (Phase 2)

Downloads PDB entries listed in a transfer manifest from the wwPDB Beta rsync
server and uploads them to an S3 staging prefix for Phase 3 promotion.

This notebook runs **locally** (where `rsync` is available) and pushes staged
files directly to MinIO — either a local test container or the production
Lakehouse via SSH tunnel. See the `TARGET` cell below for setup instructions.

**When to use this notebook vs the CTS container:**
- Use this notebook when CTS is unavailable (e.g. local development or
  one-off runs via SSH tunnel to the production Lakehouse).
- The `pdb_rsync_sync` CLI container is preserved for future CTS use.

Steps:
1. Configure `TARGET` and credentials
2. Configure bucket, manifest source, staging prefix, and worker count
3. Preview the first 10 manifest lines to verify before committing
4. Download PDB entries via rsync and upload to staging
5. Review the download/stage report


## Path formats quick reference

| Suffix in variable name | Format | Example |
|-------------------------|--------|---------|
| `_BUCKET` | bucket name only | `cts` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `staging/pdb-run1/` |
| `_S3_KEY` | S3 object key (no scheme/bucket) | `staging/pdb-run1/transfer_manifest.txt` |
| `_PATH` | local filesystem path | `output/transfer_manifest.txt` |

Staging object: `s3://{STAGING_BUCKET}/{STAGING_KEY_PREFIX}raw_data/<hash>/<pdb_id>/…`  
Report:         `s3://{STAGING_BUCKET}/{STAGING_KEY_PREFIX}download_report.json`

In [ ]:
"""Imports and S3 client initialisation."""

import json

from cdm_data_loaders.pipelines.pdb_rsync import download_and_stage
from cdm_data_loaders.utils.s3 import get_s3_client, reset_s3_client

In [ ]:
"""Configure parameters.

Provide exactly one of MANIFEST_S3_KEY (read from S3) or MANIFEST_LOCAL_PATH (read from disk).
Set the other to None.

Disk space note: downloads are pipelined — each worker downloads one PDB entry, uploads all
its files to S3, deletes the local copies, then picks up the next entry. Disk usage at any
point is bounded to approximately WORKERS × max_entry_size (typically 5–20 MB per entry).
The full archive (~100 GB) is never written to local disk.

Set LIMIT to a small number (e.g. 5) to test the workflow before a full run.
"""

# S3 bucket where the manifest lives and where staged files will be written
# format: bucket name (no s3:// scheme)
STAGING_BUCKET = "cts"

# S3 object key of the transfer manifest written by Phase 1
# format: S3 object key within STAGING_BUCKET (no scheme, no bucket)
# Set to None to use MANIFEST_LOCAL_PATH instead
MANIFEST_S3_KEY: str | None = None

# Local path to the transfer manifest (alternative to MANIFEST_S3_KEY)
# format: local filesystem path
# Set to None to use MANIFEST_S3_KEY instead
MANIFEST_LOCAL_PATH: str | None = "output/transfer_manifest.txt"

# S3 key prefix for staged output files (must match what Phase 3 expects)
# format: S3 key prefix within STAGING_BUCKET (no scheme, no bucket)
STAGING_KEY_PREFIX = "io/matt-cohere/staging/pdb-run1/output/"

# Number of parallel rsync workers
WORKERS = 4

# Maximum rsync attempts per entry (1 = no retry, 3 = up to 2 retries with exponential backoff)
RSYNC_RETRIES = 3

# Limit to first N entries (None = process all)
LIMIT: int | None = None

# Dry-run mode — download locally but skip S3 uploads
DRY_RUN = False

print(f"Bucket:           {STAGING_BUCKET}")
print(f"Manifest S3 key:  {MANIFEST_S3_KEY}")
print(f"Manifest local:   {MANIFEST_LOCAL_PATH}")
print(f"Staging prefix:   {STAGING_KEY_PREFIX}")
print(f"Workers:          {WORKERS}")
print(f"Rsync retries:    {RSYNC_RETRIES}")
print(f"Limit:            {LIMIT}")
print(f"Dry-run:          {DRY_RUN}")

In [ ]:
"""Configure S3 target and credentials.

TARGET options:
  "local"      — local MinIO test container (http://localhost:9000, minioadmin/minioadmin).
                 Start it with: docker-compose up -d  (or see pdb_e2e_walkthrough.md)

  "production" — production Lakehouse MinIO via SSH SOCKS5 tunnel.
                 One-time setup (run in a terminal before launching Jupyter):
                   ssh -f -D 1338 -N <ac.anl_username>@login.kbase.us
                   export HTTPS_PROXY=socks5h://127.0.0.1:1338   # 'h' = remote DNS resolution
                 Credentials: retrieve from the MinIO UI (https://minio.berdl.kbase.us)
                 or copy from a JupyterHub notebook:
                   import os; print(os.environ['S3_ACCESS_KEY'], os.environ['S3_SECRET_KEY'])
                 Then set MINIO_ACCESS_KEY and MINIO_SECRET_KEY in your shell environment,
                 or enter them interactively via the getpass prompt below.

  "hub"        — JupyterHub (credentials auto-injected by berdl_notebook_utils).
"""
import os

TARGET = "local"  # "local" | "production" | "hub"

if TARGET == "local":
    # Clear any proxy env vars inherited from the launch terminal (e.g. HTTPS_PROXY set
    # for an SSH tunnel) so boto3 doesn't route localhost:9000 through the SOCKS5 proxy.
    for _proxy_var in ("http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "all_proxy"):
        os.environ.pop(_proxy_var, None)
    reset_s3_client()
    get_s3_client(
        {
            "endpoint_url": "http://localhost:9000",
            "aws_access_key_id": "minioadmin",
            "aws_secret_access_key": "minioadmin",
        }
    )
elif TARGET == "production":
    import botocore.httpsession as _bch
    from urllib3.contrib.socks import SOCKSProxyManager as _SOCKSProxyManager

    # Botocore has two bugs with SOCKS proxies that require two patches:
    #
    # Bug 1: ProxyConfiguration.proxy_url_for() prepends 'http://' to any proxy URL that
    #   doesn't start with 'http', mangling 'socks5h://...' into 'http://socks5h://...'.
    #   Fix: strip the erroneous prefix back off.
    _orig_proxy_url_for = _bch.ProxyConfiguration.proxy_url_for
    def _fixed_proxy_url_for(self, url):
        result = _orig_proxy_url_for(self, url)
        if result and result.startswith("http://socks"):
            result = result[len("http://"):]
        return result
    _bch.ProxyConfiguration.proxy_url_for = _fixed_proxy_url_for

    # Bug 2: URLLib3Session._get_proxy_manager delegates to urllib3's proxy_from_url(),
    #   which only supports http/https and raises ProxySchemeUnknown for socks schemes.
    #   Fix: intercept socks URLs and use urllib3's SOCKSProxyManager instead.
    #   Important: do NOT copy pool_classes_by_scheme from the botocore session —
    #   SOCKSProxyManager needs its own SOCKS-aware connection classes (SOCKSHTTPSConnectionPool)
    #   that pass _socks_options to connections. Botocore's AWSHTTPSConnectionPool doesn't
    #   accept _socks_options and would raise TypeError.
    _orig_get_proxy_manager = _bch.URLLib3Session._get_proxy_manager
    def _socks_aware_get_proxy_manager(self, proxy_url):
        if proxy_url and proxy_url.startswith("socks"):
            if proxy_url not in self._proxy_managers:
                self._proxy_managers[proxy_url] = _SOCKSProxyManager(proxy_url)
            return self._proxy_managers[proxy_url]
        return _orig_get_proxy_manager(self, proxy_url)
    _bch.URLLib3Session._get_proxy_manager = _socks_aware_get_proxy_manager

    import getpass
    access_key = os.environ.get("MINIO_ACCESS_KEY") or getpass.getpass("MinIO access key: ")
    secret_key = os.environ.get("MINIO_SECRET_KEY") or getpass.getpass("MinIO secret key: ")
    reset_s3_client()
    get_s3_client(
        {
            "endpoint_url": "https://minio.berdl.kbase.us",
            "aws_access_key_id": access_key,
            "aws_secret_access_key": secret_key,
        }
    )
# TARGET == "hub": no explicit credentials — berdl_notebook_utils auto-injects

print(f"Target:      {TARGET}")
print(f"Proxy:       {os.environ.get('HTTPS_PROXY') or os.environ.get('https_proxy') or '(none)'}")


In [ ]:
"""Validate S3 connectivity."""

import os
import socket
from botocore.exceptions import ClientError

# Diagnostic: show proxy env vars and check tunnel port
https_proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
print(f"HTTPS_PROXY: {https_proxy!r}")

if https_proxy and "1338" in https_proxy:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _sock:
        _sock.settimeout(2)
        try:
            _sock.connect(("127.0.0.1", 1338))
            print("✓ SSH tunnel is listening on 127.0.0.1:1338")
        except (ConnectionRefusedError, TimeoutError):
            print("✗ Nothing is listening on 127.0.0.1:1338 — start the SSH tunnel first:")
            print("    ssh -f -D 1338 -N <ac.anl_username>@login.kbase.us")
            raise SystemExit(1)

s3 = get_s3_client()
try:
    resp = s3.list_objects_v2(Bucket=STAGING_BUCKET, Prefix=STAGING_KEY_PREFIX, MaxKeys=5)
    keys = [obj["Key"] for obj in resp.get("Contents", [])]
    print(f"✓ S3 connection OK — listed objects under '{STAGING_BUCKET}/{STAGING_KEY_PREFIX}':")
    for k in keys:
        print(f"  {k}")
    if not keys:
        print("  (no objects found — prefix may be empty, but bucket is accessible)")
except ClientError as e:
    code = e.response["Error"]["Code"]
    print(f"✗ S3 access check failed (HTTP {code}): {e}")
    raise


In [ ]:
"""Preview the first 10 manifest lines before committing to the full run."""

if MANIFEST_S3_KEY is not None:
    s3 = get_s3_client()
    response = s3.get_object(Bucket=STAGING_BUCKET, Key=MANIFEST_S3_KEY)
    manifest_lines = response["Body"].read().decode().splitlines()
else:
    with open(MANIFEST_LOCAL_PATH) as f:
        manifest_lines = f.read().splitlines()

data_lines = [line for line in manifest_lines if line.strip() and not line.startswith("#")]

print(f"Total entries: {len(data_lines)}")
print("First 10:")
for line in data_lines[:10]:
    print(f"  {line}")
if len(data_lines) > 10:
    print(f"  ... and {len(data_lines) - 10} more")

In [ ]:
"""Download PDB entries via rsync and upload to S3 staging."""

report = download_and_stage(
    staging_bucket=STAGING_BUCKET,
    staging_key_prefix=STAGING_KEY_PREFIX,
    manifest_s3_key=MANIFEST_S3_KEY,
    manifest_local_path=MANIFEST_LOCAL_PATH,
    workers=WORKERS,
    limit=LIMIT,
    dry_run=DRY_RUN,
)

In [ ]:
"""Display download and staging report."""

FAILURE_PREVIEW = 10

print("=" * 50)
print("DOWNLOAD & STAGE REPORT")
print("=" * 50)
print(f"Attempted:      {report['total_attempted']}")
print(f"Succeeded:      {report['succeeded']}")
print(f"Failed:         {report['failed']}")
print(f"Staged objects: {report['staged_objects']}")
print(f"Staging prefix: {report['staging_key_prefix']}")
print(f"Dry-run:        {report['dry_run']}")
print(f"Timestamp:      {report['timestamp']}")

if report["failed"] > 0:
    print("\nFailed entries:")
    for failure in report["failures"][:FAILURE_PREVIEW]:
        print(f"  {failure['pdb_id']}: {failure['error']}")
    if report["failed"] > FAILURE_PREVIEW:
        print(f"  ... and {report['failed'] - FAILURE_PREVIEW} more")

if report["dry_run"]:
    print("\nThis was a dry-run. Set DRY_RUN = False and re-run to upload to S3.")